## 传输层 · 块1 端口与复用 / 分用

> 2026-10-11 · 来源：学习对话

- **我的理解**：从 socket 视角切入传输层（原话：「如果我现在想从socket入手复习整个计算机网络」「开传输块吧」）。
- **机制层**：
  - 传输层提供**进程 ↔ 进程**的逻辑通信；网络层只到**主机 ↔ 主机**。
  - 端口 16 位：熟知 0–1023 / 登记 1024–49151 / 客户端临时 49152–65535。
  - 套接字（408 定义）＝ **(IP 地址, 端口号)**；一条 TCP 连接 ＝ **一对套接字**（四元组：源 IP、源端口、目的 IP、目的端口）。
  - **复用**（发送方，多 → 一，加端口）/ **分用**（接收方，一 → 多，按端口）。
  - 分用依据：**UDP 看 (目的 IP, 目的端口)**；**TCP 看四元组**。
- **项目映射**：Linux socket API（`socket` → `bind(IP, port)` → `connect`）、`ss` / `netstat` 看端口占用。**边界**：408 考概念与分用依据，不考 API 调用细节。
- **考试侧重点**：分用依据（UDP vs TCP）；套接字与连接标识；端口号范围分类。
- **易错点**：把「进程 ↔ 进程 / 主机 ↔ 主机」记反；TCP 分用只写目的端口（漏四元组）。
- **例题/错题**：待补充（本轮出题后回填）。
- **复习计划**：R1 @ 2026-10-12


## 传输层 · 块1 辨析：「连接」与「绑定」是两回事（补讲）

> 2026-10-11 · 来源：学习对话（用户反馈「不理解」）

- **我的理解**（原话）：「为啥有没有连接和tcp udp相关, udp不需要bind吗? 连接是connect 和listen配对」。
- **核心结论**：三个层次别混 ——（a）**协议层「连接」**＝ 双方维护的一套状态；（b）**内核层「绑定」**＝ 把本地 (IP, 端口) 登记进分用表；（c）**API 层的 `connect`** 只是一个调用，不等于连接。
- **「无连接」的含义**：UDP 无需与对端对齐任何状态（不握手、不记对方、每个报文自带完整寻址）；TCP 必须先对齐（起始序号 / 窗口 / MSS…），这个对齐过程就是三次握手 → 也就是「连接」。**连接是 TCP 为实现可靠性所建立的机制。**
- **UDP 需要 bind 吗**：**需要（服务端必须 bind）**。「无连接」≠「不用 bind」—— bind 是本地登记端口，正是块1「分用」的注册动作，与有无连接无关。客户端不显式 bind 时，由内核在**首次发送**时隐式分配临时端口（49152–65535）。
- **`connect` 的两种语义**：在 TCP 上它**触发三次握手**；在 UDP 上它**只是本地记住对端地址**（不发任何包，之后可用 `send` 代替 `sendto`，还能过滤非该源的包）。**所以 connect ≠ 有连接。**
- **`listen` / `accept`**：**TCP 专有**（被动等待的服务端侧）；UDP 无此概念。连接需要两端配合：一端 `connect` 主动发起，另一端 `listen` + `accept` 接收——用户直觉的方向是对的，但「配对」的是**两端**，不是两个调用。
- **易错点**：把「编程里的 `connect`」当成「协议层的连接」；以为 UDP「无连接」所以「不用 bind」。
- **边界**：408 只考协议层的「连接」概念与分用依据，**不考** UDP 上 `connect` 的行为。
- **复习计划**：R1 @ 2026-10-12


## 传输层 · 块1 辨析：不 bind 也有端口；服务器靠「源地址」区分对端（补讲）

> 2026-10-11 · 来源：学习对话（用户提问）

- **我的提问**（原话）：「udp客户端不bind的话, 如果我运行两个客户端, 服务器怎么区分是哪个进程发过来的呢? 游戏udp服务器还要判断端口号吗」。
- **核心澄清：两件常被混为一谈的事**
  - **分用**（内核 → 进程）：只看**目的** (IP, 端口)。服务器 bind 53，内核把目的端口 53 的报文交给它这个 socket。
  - **会话标识**（应用 → 对端）：看**源** (IP, 端口)。服务器 `recvfrom` 拿到的源地址，才是「这是哪个客户端」的唯一依据。
  - 一句话：**进来看目的端口，认人看源端口。**
- **「不 bind」≠「没有端口」**：客户端不显式 bind 时，内核在**第一次 `send`** 时**隐式绑定**一个临时端口（49152–65535）并写入分用表；此后该 socket 的所有报文都带这个源端口。**一个 socket = 一个源端口。**
- **两个客户端进程如何区分**：两个独立 socket → 内核分配**两个不同的临时端口**（同一时刻同一 (IP, 端口) 只能被一个 socket 占用，不会撞）。服务器看到的元组必然不同：或源端口不同，或源 IP、源端口都不同。
- **游戏 UDP 服务器要不要看端口号**：**要，而且这是唯一可用的会话键**。UDP 无连接、不维护连接状态表，(源 IP, 源端口) 就是「会话 ID」；回包用 `sendto(源IP, 源端口)` 原路返回。
  - 典型反例：**NAT 后多个玩家共享同一个公网源 IP**，此时**只能靠源端口**区分 —— 端口号是刚需，不是可选。
- **现实对照（边界：408 不考）**：现代实时协议（QUIC、多数自研游戏协议）还会在**应用层 payload 里再带一个 connection ID**，因为 NAT 重绑定（如 WiFi → 4G 切换导致源 IP/端口变化）不应让玩家掉线。这是「(源IP, 源端口) 作为会话键不够稳」的工程补丁。
- **考试侧重点**：ⓐ 分用依据（UDP 看目的 (IP, 端口)；TCP 看四元组）；ⓑ 「UDP 无连接」的确切含义；ⓒ 服务器为什么必须 bind。
- **易错点**：把「分用看目的端口」与「区分对端看源端口」混为一谈；以为「不 bind」=「没有端口」；以为 UDP 服务器「不需要看端口号」。
- **边界**：408 不考隐式绑定的实现细节、不考 NAT 重绑定与 connection ID。
- **复习计划**：R1 @ 2026-10-12


## 传输层 · 块1 辨析：`send/recv` 与 `sendto/recvfrom` 怎么选（补讲）

> 2026-10-11 · 来源：学习对话（用户提问）

- **我的提问**（原话）：「现在我没搞懂 send recv, sendto recvfrom, 这几个指令都是客户端服务器socket随便用的吗?」
- **核心结论**：**用哪组与「客户端/服务端」角色无关**，只看这个 socket **有没有「默认对端」**。
  - **有默认对端**（socket 状态里存了对端地址）→ `send` / `recv`（调用不带地址）。
  - **没有默认对端** → `sendto` / `recvfrom`（每次现场给目标地址 / 收回源地址）。
- **「对端」从哪来**：`connect()`（TCP 客户端；UDP 上只记地址、不发包）或 `accept()` 返回的新 fd（TCP 服务端）。
- **四象限**：

| 角色 | 协议 | 何时具备对端 | 收发用 | 归属块 |
|---|---|---|---|---|
| 客户端 | TCP | `connect` 之后 | `send` / `recv` | 1 |
| 服务端 | TCP | `accept` 返回的**新 fd** | `send` / `recv` | 1（连接管理，块3 展开） |
| 服务端 | UDP | `bind` 后（不 connect） | `recvfrom` / `sendto` | 1 |
| 客户端 | UDP | 不 connect | `sendto` / `recvfrom` | 1 |

- **三个易绕晕的点**：
  - **TCP 服务端收发用的是 `accept` 返回的新 socket**，不是 `listen` 那个；监听 socket 只负责 `accept`，从不 send/recv。
  - **UDP 服务端必须用 `recvfrom`**（用 `recv` 会丢掉源地址，无法回包）；回包用 `sendto(源IP, 源端口)` 原路返回 —— 即「进来看目的端口，认人看源端口」。
  - **UDP 一旦 `connect` 过就能改用 `send/recv`**（connect 只是本地记住对端并过滤非该源的包），所以「用哪组」与 TCP/UDP 也不是死绑的。
- **设计层**：TCP「先建通道、两端固定」→ 对端存进 socket → `send/recv` 够用；UDP 无连接、一个 socket 对着 N 个对端 → 地址无处存 → 必须每次现场给/收。**`sendto/recvfrom` 不是 UDP 专用命令，而是「地址没地方存」时的通用形态。**
- **边界**：408 考纲**不考这些 API 调用**；只考概念层（分用依据、TCP 面向连接的含义、一个 listen socket 派生多条连接）。此条仅作直觉锚定。
- **考试侧重点**：ⓐ TCP 服务端「监听 socket vs 连接 socket」的结构；ⓑ UDP 服务端如何区分并回复多个客户端（复用/分用的 API 落地)。
- **易错点**：以为「客户端用一组、服务端用一组」；以为 `sendto/recvfrom` 是 UDP 专属；把 API 层的 `connect` 当成协议层的连接（块1 已辨析）。
- **复习计划**：R1 @ 2026-10-12


## 传输层 · 块1 辨析：UDP 服务端要自己保存客户端 sockaddr 吗（补讲）

> 2026-10-11 · 来源：学习对话（用户提问）

- **我的提问**（原话）：「那 udp 服务器得手动保存所有收到的客户端 sockaddr?」
- **核心结论**：**看你要不要「记住这个客户端」**。UDP 服务端没有保存义务。
  - **① 纯请求-响应（无状态）→ 不用存**：`recvfrom` 拿到的 `sockaddr_in` 是**栈上局部变量**，本轮用它 `sendto` 原路回包，随栈帧消失。DNS / NTP / echo 皆如此。
  - **② 需要会话状态 → 必须自己存**：内核只按目的端口分用、**不认人**；要区分/记住客户端，须自建以 **(源IP, 源端口)** 为 key 的表。
- **代码对照**：

```c
// 无状态：回完即弃
for (;;) {
    len = sizeof cli;
    ssize_t n = recvfrom(fd, buf, sizeof buf, 0, (struct sockaddr*)&cli, &len);
    sendto(fd, buf, n, 0, (struct sockaddr*)&cli, len);
}

// 有状态：自建会话表
unordered_map<Endpoint, PlayerState> players;   // key = IP(4B)+port(2B)
for (;;) {
    len = sizeof cli;
    n = recvfrom(fd, buf, sizeof buf, 0, (struct sockaddr*)&cli, &len);
    Endpoint who = to_key(cli);
    if (!players.count(who)) players[who] = PlayerState{};
    players[who].last_seen = now();
    sendto(fd, reply, rlen, 0, (struct sockaddr*)&cli, len);
}
```

- **实现细节**：接地址用 `struct sockaddr_storage`（128B，IPv4/IPv6 通吃），别用 16B 的 `struct sockaddr`；key 是定长字节串（IPv4 = 4B IP + 2B port）。
- **设计层（为什么内核不存）**：这正是**「无连接/无状态」的定义**——内核不替应用维护会话表。好处：一个 socket 扛上百万客户端、不吃内核连接表；代价：会话维护全落到应用。TCP 相反（内核每条连接存一份四元组 + 一个 fd，方便但受内核资源限制）。**UDP 的可扩展性，代价就是应用自己维护状态。**
- **结构对比（值得记）**：TCP 服务端 N 条连接 = N 个 `accept` 出来的 fd；**UDP 服务端永远是那一个 socket**。
- **现实补丁**：NAT 重绑定（WiFi→4G 源 IP/端口变化）会让 (IP, port) 认不出老玩家 → QUIC / 现代游戏协议在**应用层 payload 带 connection ID**，作更稳的会话键。
- **边界**：408 考「UDP 无连接/无状态」的含义、UDP 一个 socket 服务多客户端、分用依据；**不考** sockaddr 结构与 API 细节。
- **易错点**：以为「UDP 无状态」=「应用层也不能有状态」（游戏服务器是反例）；以为 `recvfrom` 的地址会被自动保存（不会）。
- **复习计划**：R1 @ 2026-10-12


## 传输层 · UDP 的正确定位：「白板」不等于「保证质量」

> 2026-10-12 · 来源：学习对话（用户自述判定）

- **我的理解**（原话）：「tcp 就是打样版的可靠连接, udp 则是提供的最大性能的白板, 只保证到手包的质量和速度」。
- **判定：「白板」这个比喻对，「保证到手包的质量」说反了。**
- **UDP 实际只做两件事**：① **复用 / 分用**（端口）；② **差错检测**（可选校验和）。此外什么都不做。
- **UDP 不保证**：送达、不丢、不重复、不乱序、不保证时延。**「质量」恰恰是它放弃的东西。**
- **与链路层同一个哲学**：UDP 校验和也是**只检不修**——检出错就**丢弃**，不重传。**「发现错误就丢，修复交给能负责端到端的那一层」是全栈一致的思路。**
- **正确的对比表述**（408 常考的四维辨析）：

| 维度 | TCP | UDP |
|---|---|---|
| 连接 | 面向连接（三次握手） | 无连接 |
| 可靠 | 确认 + 重传，可靠 | 不保证 |
| 有序 | 保序（按字节流重组） | 不保序 |
| 开销 | 首部 20B + 状态机 + 窗口 / 定时器 | 首部 8B，无状态 |
| 传输单位 | **面向字节流**（无边界） | **面向报文**（保边界） |

- **准确的定位说法**：UDP 换来的不是「更好的性能」，而是**最小的开销与自由**——它把「可靠性做到什么程度」的决定权**交给应用层**（实时音视频宁可丢帧也不要重传时延；DNS 一次一问一答，超时重传由应用自己做）。
- **「打样版」措辞纠正**：TCP 不是「样板」，它是**内核把可靠性做成了通用服务**；UDP 则是**只提供最小骨架，其余由应用自己搭**。
- **考试侧重点**：UDP 首部字段、面向报文 vs 面向字节流、UDP vs TCP 四维辨析。
- **易错点**：说「UDP 保证质量」；把 UDP 说成「TCP 的低质量版」（两者是**职责不同**，不是质量高低）；忘了 UDP 面向报文、TCP 面向字节流。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13


## 传输层 · TCP 的时延比 UDP 高在哪，以及「时延 ≠ 可靠性」

> 2026-10-12 · 来源：学习对话（用户自述判定）

- **我的理解**（原话）：「tcp 的时延肯定比 udp 高吧, 而且时延又不是可靠性」。
- **判定**：「时延不是可靠性」**完全正确**（二者正交）；「TCP 时延肯定更高」**半对**——需要加条件。
- **精确表述**：TCP 提高的是**时延的上界与抖动（tail latency）**，**不是中位数**。稳态无丢包、连接已建立、cwnd 已长满时，TCP 与 UDP 的单包时延**几乎相同**（都是 处理 + 传输 + 传播 + 排队）。
- **TCP 额外时延的四个来源（均为条件性）**：
  1. **建连（三次握手）**：1 RTT —— 每条新连接一次；短连接里占比极高。
  2. **重传**：丢包后等 RTO（超时，百 ms ~ s 级）或快重传（约 1 RTT）—— **P99 尾时延的主源**。
  3. **cwnd 限速**：慢开始爬坡期速率受限；丢包后 cwnd 减半、速率骤降。
  4. **Nagle / 延迟确认**：小包合并等待，几十 ms 级（408 不考，但能解释现实卡顿）。
- **关键结论**：**TCP 的时延代价来自「可靠必须重传」和「拥塞控制必须试探」，不来自「确认」本身** —— ACK 可以捎带、可以流水线，它不阻塞数据。
- **「时延 ≠ 可靠性」：两个正交维度**
  - **可靠性 = 正确性**：不丢 / 不重 / 不乱序 —— **离散、二值**（到了 / 没到）。
  - **时延 = 时效性**：多久到 —— **连续、有分布**（中位数 / P99 / 抖动）。

| | 低时延 | 高时延 |
|---|---|---|
| **可靠** | TCP + 大 cwnd + 高质量线路（内网 RPC） | TCP over 卫星（RTT 500 ms+）/ 丢包多要重传 |
| **不可靠** | UDP + 实时音视频（丢帧也不重传） | UDP + 拥塞（排队暴涨，且无重传兜底） |

- **设计层**：**「可靠」与「低时延」是冲突目标** —— TCP 为了可靠性会主动做增加时延的事（重传、等 RTO）；UDP 放弃可靠性正是为摆脱这个约束。**这才是实时音视频选 UDP/RTP 的根本原因**（不是「UDP 质量差所以快」）。
- **考试侧重点**：TCP/UDP 对比（面向连接 / 可靠 / 开销 / 面向字节流 vs 面向报文）；**时延四分量（发送 / 传播 / 处理 / 排队）的计算与辨析** ★核心。
- **易错点**：把「TCP 时延一定高」当无条件结论（稳态下中位数几乎相同）；把时延算进「可靠性」维度；混淆「发送时延」与「传播时延」。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13

## 传输层 · 拥塞控制只看丢包吗：经典 TCP 是，但不是唯一

> 2026-10-12 · 来源：学习对话（用户提问）

- **我的提问**（原话）：「拥塞控制是只用丢包不看时延的波动啥的吗」。
- **核心答案**：**经典 TCP（Tahoe / Reno / NewReno，也就是 408 只考的那套）确实只用丢包作信号**，不看时延。但这是「一种选择」，**不是拥塞控制的定义**。
- **经典 TCP 的两个信号（都源于丢包）**：
  - **超时** → 严重拥塞：`ssthresh = cwnd/2`，`cwnd = 1`，重新慢开始。
  - **三个重复 ACK** → 轻度拥塞：快重传 + 快恢复，`ssthresh` 与 `cwnd` 减半。
- **为什么不用时延（四条硬伤）**：① **噪声大**（RTT 受路径变化 / 队列抖动 / 无线重传 / OS 调度影响，绝对时延难设阈值）；② **基线难定**（需「最小 RTT」作参照，路径一变基线就废）；③ **丢包信号干脆**（二值、明确，对「队列满」基本无误判）；④ **公平性**（同一信号才互相公平，换信号会互相压制）。
- **存在的非丢包信号**：

| 方案 | 信号 | 性质 | 状态 |
|---|---|---|---|
| Reno / NewReno（408 考） | 丢包 | **事后**（结果） | 主流基线 |
| TCP Vegas | RTT 相对 baseRTT 的增长 | **事前**（前兆），预防式 | 未普及 |
| LEDBAT | 排队时延 | 让路给前台流量 | 特定场景（后台下载） |
| ECN / DCTCP | 路由器队列将满时**标记**（不丢） | **显式**（第三种信号） | 逐步推广 |
| BBR | 带宽 × 最小 RTT 模型 | 建模，不直接看丢包 | 谷歌 / YouTube 部署 |

- **核心洞察**：**丢包是拥塞的「结果」（事后响应），时延是拥塞的「前兆」（事前预防）**。Vegas 能更早减速，但与「只看丢包」的 TCP 竞争时**吃亏**（它先让速、别人先占带宽）—— 在全员只看丢包的环境里，谁先礼貌谁先饿死，这就是 Vegas 未普及的根本原因。ECN 是折中（要路由器配合标记，不靠猜）。
- **「只看丢包」的固有代价**：**TCP 无法区分丢包原因** —— 链路误码与缓存溢出对它是一样的。无线链路误码率高，会被误判为拥塞 → 无谓降窗。这是该方案的绕不过去的缺陷，也是无线场景要专门优化 TCP 的原因。
- **408 边界**：只考 Tahoe / Reno / NewReno 的**丢包式**（慢开始 / 拥塞避免 / 快重传 / 快恢复）；**Vegas / BBR / ECN / LEDBAT / DCTCP 一律不考**。知道它们存在只是为明白「408 教的是一套特定设计，不是唯一解」。
- **易错点**：以为「拥塞控制就是看丢包」是定义；把「超时」与「三个重复 ACK」的响应弄反（超时 = 严重、三重复 ACK = 轻度）。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13

## 传输层 · 块2 UDP 与 TCP 概述（首部与「字节流 vs 报文」）

> 2026-10-12 · 来源：学习对话

- **我的指令**（原话）：「开始 tcp udp 吧, 先顺着把拥塞控制慢启动这些说完吧」。
- **必考点清单**：① TCP 首部字段含义与长度（固定 20B / 最长 60B）★；② UDP 首部四字段（8B）★；③ 序号 / 确认号语义与计算 ★；④ 面向字节流 vs 面向报文 ★；⑤ 标志位作用（SYN / FIN / ACK / RST / URG / PSH）☆。

### 机制层（①）

**UDP 首部（固定 8 字节，四字段）**

| 字段 | 说明 |
|---|---|
| 源端口 (16) | **可选** —— 不需要对方回信时可全 0 |
| 目的端口 (16) | 必须；**分用靠它** |
| 长度 (16) | 首部 + 数据，**最小值 8**（只有首部） |
| 校验和 (16) | 检验**首部 + 数据**（另加 12B 伪首部）；检出错即丢弃，不重传 |

**TCP 首部（固定 20 字节，含选项最长 60 字节）**

| 字段 | 位数 | 要点 |
|---|---|---|
| 源端口 / 目的端口 | 各 16 | 分用；TCP 分用看**四元组** |
| 序号 seq | 32 | **本报文段数据第一个字节**在字节流中的编号（不是「第几个报文段」） |
| 确认号 ack | 32 | **期望收到的下一个字节**的序号 = 已收到最后一个字节 + 1 |
| 数据偏移 | 4 | 首部长度，**单位 4B**，取值 5–15 → 首部 **20–60B** |
| 保留 | 6 | 未用 |
| 标志 | 6 | URG / ACK / PSH / RST / SYN / FIN（408 高频：**ACK / SYN / FIN**） |
| 窗口 rwnd | 16 | 接收窗口，**单位 1B**（不缩放时最大 65535） |
| 校验和 | 16 | 检验首部 + 数据（+ 伪首部），**也是只检不修** |
| 紧急指针 | 16 | 配合 URG，408 低频 |
| 选项 | 可变 | 最长 40B（MSS / 窗口扩大 / SACK 等） |

- **两个必背长度**：UDP 首部固定 **8B**；TCP 首部固定 **20B**、含选项最长 **60B**。
- **数据偏移为何是 4 位**：4 位最大 15，15 × 4 = 60，**刚好覆盖最长首部** —— 这是设计不是巧合。

**面向字节流 vs 面向报文（★必考）**

- **UDP 面向报文**：应用交一个报文，UDP 加首部就发；收方**一次收一个完整报文**，**边界保留**（长度字段就是为此）。
- **TCP 面向字节流**：应用写多少字节，TCP 只当**无边界字节序列**；由 TCP 自行切段（可能合并两次 `send`，也可能拆开一次 `send`）。
- **后果（考法）**：TCP 上应用必须**自己做消息边界**（长度前缀 / 分隔符 / 定长），否则收方不知一条消息到哪结束（即「粘包」问题）；UDP 天然保边界，不用做。

### 设计层（②）

- **UDP 只要 4 个字段的判据**：只保留「最小骨架」必需的 —— 端口（分用）、长度（报文边界）、校验和（自检）。没有序号（不重排）、没有窗口（不做流控）、没有标志位（无连接可管理）。
- **TCP 首部多出的字段 = 它多提供的服务**：序号 / 确认号 → 可靠 + 有序；窗口 → 流量控制；SYN / FIN / RST → 连接管理；数据偏移 → 首部可变长（装选项）；校验和 → 端到端检错。**首部是「服务清单」的镜像。**
- **现实对照**：Stanford CS144 的 `cs144/src/byte_stream.cc`（**仓库名 cs144 + 文件路径**）就是「TCP 面向字节流」这个抽象的裸实现 —— 只提供 `write()` / `read()` 与字节计数，**完全不认识消息边界**，正好实证上面 2.3 的结论；`cs144/src/reassembler.cc` 处理乱序字节段重排（对应块4 可靠传输）。**边界**：这两段代码只用来锚定「字节流 / 乱序重组」的直觉，408 只考纸面推演不考实现；且**该仓的 TCP sender / receiver（含拥塞控制）尚未实现，该部分代码尚无**。

### 验收层（③）

| # | 题型 | 模板要点 | 优先级 |
|---|---|---|---|
| 1 | TCP 首部字段含义 / 长度 | 记住 20B 固定、60B 上限；数据偏移 × 4 = 首部字节数 | ★核心 |
| 2 | UDP 首部四字段 | 源端口 / 目的端口 / 长度 / 校验和，共 8B | ★核心 |
| 3 | 序号 / 确认号计算 | ack = 期望收到的下一个字节序号；SYN / FIN 各占 1 个序号 | ★核心 |
| 4 | 字节流 vs 报文 | UDP 保边界、TCP 不保；粘包问题归因到 TCP | ★核心 |

- **易错点**：说「TCP 保证消息不乱序」—— 错，TCP 保证的是**字节流**不乱序，不保证「消息」这个单位（它根本不知道消息边界）；把「序号」理解成「第几个报文段」；混淆「数据偏移」与「窗口」的单位（前者 4B，后者 1B）。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13

## 传输层 · 块5 拥塞控制（慢开始 / 拥塞避免 / 快重传 / 快恢复）★核心

> 2026-10-12 · 来源：学习对话

- **我的指令**（原话）：「先顺着把拥塞控制慢启动这些说完吧」。
- **必考点清单**：① cwnd 序列推演 ★；② 给定事件求 cwnd / ssthresh ★；③ 累计发送量（纯慢开始 n 个 RTT 共发 $2^n - 1$ 个报文段）★；④ 曲线判读 ☆；⑤ 识别算法 ☆。

### 机制层（①）

**三个窗口变量（不分清后面全乱）**

| 变量 | 谁维护 | 含义 |
|---|---|---|
| rwnd（接收窗口） | **接收方通告** | 我还能收多少（流量控制） |
| cwnd（拥塞窗口） | **发送方估计** | 网络还能承受多少（拥塞控制） |
| 发送窗口 | 发送方取 | $\min(\text{rwnd},\ \text{cwnd})$ |

- **关键**：cwnd 是**猜出来的量** —— 网络不会告诉发送方「还能承受多少」，只能靠**观察反馈（丢包）去试探**。整个拥塞控制就是一套试探算法。
- 初始值：cwnd = **1 MSS**；ssthresh 为慢开始门限（分界线）。

**四个算法**

| 算法 | 规则 | 效果 |
|---|---|---|
| **慢开始** | 每收到一个 ACK，cwnd += 1 MSS | 每 RTT **翻倍**（指数）；直到 cwnd ≥ ssthresh 转拥塞避免 |
| **拥塞避免** | 每收到一个 ACK，cwnd += 1/cwnd MSS | 每 RTT **+1**（线性）；直到检测到拥塞 |
| **快重传** | 收到 **3 个重复 ACK** 立即重传丢失段 | 不等 RTO，省一个超时等待 |
| **快恢复** | 不回到慢开始，cwnd 设为新 ssthresh | 直接进拥塞避免 |

- **「慢开始」名字的坑**：指**起点慢**（从 1 开始），不是增长慢 —— 它是最快的（指数）增长。
- **「为什么是 3 个重复 ACK」**：1 个可能只是乱序；连续 3 个说明后面的段都到了，那一段是**真丢了**。工程经验值，非数学结论。

**两类拥塞事件的响应不同（必考核心对比）**

| 拥塞事件 | 严重度 | ssthresh | cwnd | 之后走 |
|---|---|---|---|---|
| **超时**（RTO 到期） | 严重 | $\text{cwnd}/2$ | **1** | **慢开始** |
| **3 个重复 ACK** | 轻度 | $\text{cwnd}/2$ | **ssthresh**（新值） | **拥塞避免** |

- 超时意味着**连 ACK 都没回来**（可能整窗丢失、网络很糟）→ 退到底重来；3 个重复 ACK 说明**数据仍在流动**（只是破了个洞）→ 只退一半。

### 设计层（②）

- **为什么先指数后线性**：远离容量时要**尽快吃满带宽**（指数快速逼近）；接近容量时要**小心别冲过头**（线性缓步，减小过冲代价）。全程指数 → 一次性大批丢包；全程线性 → 爬得太慢。这是速度与代价的折中。
- **为什么区分超时与重复 ACK**：两者**携带的信息量不同**（有无 ACK 回流 = 网络是否还在工作）。
- **AIMD 与公平性（最值钱的一条）**：机制本质是 **AIMD = 加法增大（AI）+ 乘法减小（MD）**。多条 TCP 流竞争同一瓶颈时，各自线性膨胀，谁先触顶谁先减半让路，其余继续膨胀 → 反复后**收敛到均分带宽**。**AIMD 被选中不是因为快，而是因为既稳定又公平。**
- **流量控制 vs 拥塞控制（再钉死一次）**：

| | 流量控制 | 拥塞控制 |
|---|---|---|
| 保护谁 | **接收方**（别冲爆其缓存） | **网络中间**（别压垮路由器） |
| 窗口 | rwnd | cwnd |
| 怎么得到 | 接收方**显式通告** | 发送方**隐式试探**（靠丢包推断） |
| 合起来 | 发送窗口 $=\min(\text{rwnd},\text{cwnd})$ | |

- **现实对照**：Linux 拥塞控制是 `net/ipv4/tcp_cong.c` 里的可插拔模块（默认 CUBIC），可切换。**边界**：408 只考 Reno 系（慢开始 / 拥塞避免 / 快重传 / 快恢复）的纸面推演，**不考** CUBIC / BBR 等现代算法实现。

### 验收层（③）

| # | 题型 | 模板要点 | 优先级 |
|---|---|---|---|
| 1 | **cwnd 序列推演** | 按 RTT 逐个推进；每步先判阶段（慢开始 ×2 / 拥塞避免 +1），再套规则 | ★核心 |
| 2 | **给定事件求 cwnd / ssthresh** | 超时 → ssthresh = cwnd/2、cwnd = 1；3 重复 ACK → ssthresh = cwnd/2、cwnd = ssthresh | ★核心 |
| 3 | **累计发送量** | 纯慢开始 n 个 RTT 共发 $1+2+4+\cdots+2^{n-1} = 2^n - 1$ 个报文段 | ★核心 |
| 4 | 曲线判读 | 陡升 = 慢开始；缓升 = 拥塞避免；垂降到底 = 超时，垂降一半 = 重复 ACK | ☆常规 |
| 5 | 识别算法 | 给行为描述反问算法名（或反向） | ☆常规 |

**代表例**：TCP 连接刚建立，cwnd = 1 MSS，ssthresh 初值 = 8 MSS，过程中不丢包。
（1）cwnd 随 RTT 如何变化？（2）第 4 个 RTT 结束前共发送多少报文段？（3）若第 12 个 RTT 结束时**超时**，新的 ssthresh 与 cwnd？

**解**：（1）RTT1→1、RTT2→2、RTT3→4、RTT4→**8** = ssthresh，此后转拥塞避免：RTT5→9、…、RTT12→**16**。（2）$1+2+4+8 = 15$ 个（$= 2^4 - 1$）。（3）超时时 cwnd = 16 → ssthresh = $16/2$ = **8 MSS**，cwnd = **1**，重新慢开始。

**变式**：把（3）的「超时」换成「收到 **3 个重复 ACK**」—— ssthresh 仍为 **8**，但 **cwnd = 8**（不是 1），**直接进拥塞避免**（下一 RTT 为 9）。这说明**同一个丢包事件，慢恢复要十几个 RTT 才回到原位，快恢复只要几个**。

- **易错点**：① 慢开始算成「每 RTT +1」（那是拥塞避免）；拥塞避免算成「每 ACK +1」（那是慢开始的规则，拥塞避免是每 RTT +1 / 每 ACK 加 $1/\text{cwnd}$）；② **超时后 cwnd 必须回到 1**，不是 ssthresh；③ ssthresh 非 2 的幂时慢开始翻倍会「越过」它（如 4 翻倍到 8 而 ssthresh = 6）→ **以翻倍后的值进入拥塞避免**，考场若图已给出**一律以图为准**。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13

## 传输层 · 慢开始的真相：ACK 自时钟、事件驱动状态机与内核实现（补讲）

> 2026-10-12 · 来源：学习对话

### ① 核心纠正：慢开始不是「发一个等一个」

- **误解**：发 1 个包 → 等一个 RTT 的 ACK → 再发下 1 个包。若真如此，吞吐被 RTT 锁死（1 包/RTT），TCP 不可能跑满带宽。
- **真相**：**ACK 自时钟（ACK clocking）**。`cwnd` 的本质是「**在途（已发未确认）字节的额度**」。额度多大，就能一口气连续压进多少个报文段，管道瞬间填满；每回来一个 ACK，额度释放一格并 +1，立刻又能补发新的。
- 一个 RTT 内：cwnd 个报文段**同时在途** → 全部被确认 → cwnd 翻倍。即 1 → 2 → 4 → 8 …（这正是「每 RTT 翻倍」的由来）。
- **只有第一拍**（cwnd = 1）字面意义上是「发一个等一个」；即便那一拍，进程也不阻塞——`write()` 只把数据丢进 socket 发送缓冲区就返回，其余全由内核异步推进。
- cwnd 的物理含义 = **带宽时延积（BDP = 带宽 × RTT）** 的估计值；慢开始就是用指数增长去逼近它。

### ② 「探测」的准确说法

- 不是探「通道有没有人在用」，而是探「**这条路径最多能承受多少在途量**」。
- 判据：**一整批在途包都被 ACK 确认** → 容量尚未到顶 → 可以翻倍。
- **每条新连接都要重新爬一遍**：经典 cwnd 初值 1 MSS；现代按 RFC 6928 提到 **IW10**（10 个 MSS）。
  - 长连接：启动成本被摊薄。
  - 短连接（每个请求新建一条）：反复吃慢开始的亏 → 这是 keep-alive / 连接池 / HTTP/2 多路复用的根本动机之一。

### ③ 流程复核（三条全对）

| # | 事件 | ssthresh | cwnd | 之后 |
|---|---|---|---|---|
| 1 | cwnd 涨到 ssthresh | 不变 | 继续涨 | 切**拥塞避免**（每 RTT +1） |
| 2 | **3 个重复 ACK** | cwnd/2 | = 新 ssthresh | **快重传 + 快恢复** → 拥塞避免 |
| 3 | **严重超时（RTO）** | cwnd/2 | **1** | **重新慢开始** |

### ④「割裂感」的收拢：事件驱动的状态机 + AIMD

割裂感来自三者的**触发频率完全不同**：

- 慢开始 / 拥塞避免 —— **每个字节都在跑**（常规巡航，稳态）
- 快重传 / 快恢复 —— 每次**轻度**丢包（异常处理，预测式）
- 超时重传 —— 每次**严重**丢包（兜底，最后防线）

**统一视角**：这是一台**事件驱动的状态机**。状态 =（cwnd, ssthresh, 当前阶段）；事件 =（ACK 到达 / 3 个重复 ACK / 超时）。所谓「割裂」其实是**三类事件三种响应**——不是三套独立机制，而是**一张事件处理表**。

再统一一层：**AIMD**（Additive Increase / Multiplicative Decrease）——平时 +1（AI），遇事件砍半（MD）。整套拥塞控制就是这一条策略在三种严重度下的展开。

### ⑤ 实现位置：内核 TCP 栈（用户态完全不可见）

- **发送路径**：应用 `write()` → socket 发送缓冲区 → `tcp_sendmsg` 切 skb → **`tcp_write_xmit`（受 in-flight < min(cwnd, rwnd) 约束）** → qdisc 排队 → 网卡驱动 ring buffer → DMA → 网卡硬件。
- **ACK 返回路径**：网卡 → DMA → 硬中断 → 软中断（NAPI）→ `tcp_v4_rcv` → **`tcp_ack()` 更新 cwnd / ssthresh、清掉已确认的在途量** → **立刻**调 `tcp_write_xmit` 补发新包。
- **关键**：ACK 一回来，内核当场把释放出的额度换成新包发出，**不需要应用再 write、也不需要等整批确认**——这就是「自时钟」：ACK 的到达节奏驱动发送节奏。
- 拥塞控制是**可插拔模块**（`net/ipv4/tcp_cong.c`，`tcp_register_congestion_control`）；Linux 默认 **CUBIC**，另有 Reno / Vegas / BBR。
  - ⚠️ 注意：**408 考的是 Reno 系**（拥塞避免阶段线性 +1）；CUBIC 的拥塞避免是三次函数，不是线性。
- **408 边界**：只考协议行为（cwnd 曲线推演），不考内核实现；上述文件名仅作直觉锚点。

### ⑥ 易错点

- ❌ 把慢开始理解成**串行发送**（发一个等一个）——它是**流水线**，cwnd 个包同时在途。
- ❌ 以为「收齐 n 个 ACK 才敢发下 n 个」——每个 ACK 单独到达即可立刻补发，不等整批。
- ❌ 把「每 RTT 翻倍」的**微观规则**说成「每 ACK +1 个报文段」——微观是「每 ACK +1 额度」，宏观效果才是翻倍。


## 传输层 · 发送是「水位驱动」而非「忙等轮询」：cwnd 是上限不是每批量（补讲）

> 2026-10-12 · 来源：学习对话

### ① 核心纠正

- **误解 A**：「用户写内存 → 内核让网卡发，整个过程相当于 `while !empty` 忙等去发」。
  - 内层确实有个 `while`（会一直发到窗口顶，即「拉满」），但**外层是事件触发，不是忙等轮询**。内核只在四类事件发生时才尝试发送：
    1. 应用 `write()` 写入新数据
    2. **ACK 到达**（释放额度）
    3. 重传定时器到期
    4. 网卡腾出 TX 描述符
  - 静止时它**一动不动**，不占 CPU —— 这是「忙等」说法最不对的地方。
- **误解 B**：「每次只能发送 cwnd 大小的内容」。
  - cwnd **不是「每批发送量」，而是「在途（已发未确认）总量的上限」**，是一条**水位线**。发送是**滑动、增量**的：发多少、ACK 回来释放多少、就补发多少。

### ② 正确的循环（伪代码）

```
事件触发（应用 write / ACK 到达 / 重传超时 / 网卡腾出描述符）:
    while 有数据待发
          and 在途字节数 < min(cwnd, rwnd)
          and 网卡 TX 描述符还够用:
        切出不超过 MSS 的一段，加首部交给网卡
        在途字节数 += 段长
    在途量顶到上限 → 退出循环，静止等待下一个事件
```

- **内层 `while`** 对应「拉满」直觉；**外层事件**保证不 spin。
- 准确概括：**事件触发 + 水位驱动**。

### ③ 三层上限（"每次能发多少"取最小）

| 上限 | 谁定的 | 含义 |
|---|---|---|
| **cwnd** | 发送方估计 | 网络能承受多少 |
| **rwnd** | 接收方通告 | 对方缓存还能收多少 |
| **TX ring / qdisc** | 网卡与内核 | 硬件与内存的物理队列还有几格 |

完整说法：**任意时刻在途字节 ≤ min(cwnd, rwnd, 网卡队列容量)**。

### ④ 「100% 拉满」分三层，不能混

| 拉满什么 | 成立吗 | 说明 |
|---|---|---|
| **CPU 拉满** | ✗ | 发送动作就是填几个描述符，填完立刻返回；TCP 发送路径**没有忙等循环** |
| **在途量拉满** | ✓ | 窗口允许且数据充足时，内核一直发到在途 = 窗口，然后**真的停**（一个包都不发） |
| **链路跑满** | 不一定 | cwnd < 带宽时延积时链路是**饿的**——此时是**发送方在等自己涨窗口**，不是链路在等发送方 |

- 第三行是关键：**慢开始阶段"发得慢"的瓶颈既不在链路也不在 CPU，而在 cwnd 这个人为上限**。
- 稳态吞吐 ≈ **cwnd / RTT** —— 所以 cwnd 直接决定吞吐。

### ⑤ 比喻：不是水泵，是水闸

- 上游水位 = **待发数据**（socket 发送缓冲区还剩多少）
- 闸门上沿 = **cwnd**（在途量上限）
- 放水 = **发一个段**
- ACK 回来 = **下游水位降一格**，闸门再放一格

闸门只在「下游水位低于上沿」时放水；放满即**静止**，等 ACK 这个事件把水位拉下来再动。

### ⑥ 判定表

| 说法 | 判定 |
|---|---|
| 用户把发送内容写到内存里 | ✓ 准确说写进**内核 socket 发送缓冲区**（默认是 copy；`MSG_ZEROCOPY` / `sendfile` 才零拷贝） |
| 内核指定网卡发送这块内存 | ✓ 内核把 skb 数据**地址写进网卡 TX 描述符环**，网卡自己 DMA 读内存 |
| 相当于 `while !empty` 尽力发 | ✓ 内层循环确实如此（顶到窗口为止） |
| **每次只能发 cwnd 大小** | ✗ 应为「**任意时刻在途量 ≤ cwnd**」——水位线，不是每批量 |
| **是忙等 / 100% 占 CPU** | ✗ 是**事件触发**，静止时不耗 CPU |

### ⑦ 现实补充（408 不考）

- Linux 有 **TSO / GSO**：内核可能一次交给网卡几十 KB 的「超级段」，由**网卡硬件**切成 MSS 大小的线速帧。所以「一次交多少」的实际粒度比 MSS 粗，但窗口的约束不变。

### ⑧ 易错点

- ❌ 把 cwnd 当成「每批发送量」——它是**在途量上限**。
- ❌ 把发送路径想成忙等循环——它是**事件驱动**，静止时零开销。
- ❌ 认为「TCP 发送就是把链路跑满」——窗口小于 BDP 时链路是饿的，瓶颈在窗口。


## 传输层 · CS144 的 TCPSender：ACK 触发的是 fill_window，不是「一个包」（补讲）

> 2026-10-12 · 来源：学习对话（对照 CS144 lab3 / lab4）

### ① 直接回答

- 收到 ACK 是**触发源之一**，但触发的动作不是"发一个包"，而是调用 **`fill_window()`**——它内部是 **`while` 循环**，把窗口腾出的空间**一次填满**，发出 **0 个 / 1 个 / 多个**段。
- 所以「1 个 ACK ↔ 1 个新包」**不是规则**。只有"该 ACK 恰好释放 1 MSS、且窗口其余部分早已填满"时才呈现 1:1 的假象。
- **累积 ACK** 最容易打破 1:1：一个 ACK 可一次确认多个段 → 在途量骤降多格 → 一次 `fill_window()` 连补多段。

### ② fill_window 的语义（CS144 lab3）

每轮循环取一个段，长度 = **min(**

- `TCPConfig::MAX_PAYLOAD_SIZE`
- `window_size − bytes_in_flight()`
- `stream_in().buffer_size()`

**)**，直到窗口填满、或流已空 / 已发 FIN。

- 首轮若 `_next_seqno == 0` 则置 `SYN`；流 EOF 且窗口够则置 `FIN`。
- 首个未确认段发出时**启动 RTO 定时器**（一批段共享同一个定时器）。

### ③ 四个触发入口（「事件驱动」在 CS144 里的具体长相）

| 入口 | 场景 |
|---|---|
| `TCPConnection::write()` | 应用写入新数据 |
| `TCPConnection::end_input_stream()` | 主动关闭（发 FIN） |
| `TCPConnection::segment_received()` | 收到任何报文段（含 ACK / 数据） |
| `TCPConnection::tick()` | 定时驱动（RTO 超时重传） |

四者最终都汇到 **`fill_window()`** —— 一个统一的「把窗口填满」循环。

### ④ 三个必须知道的特例

- **零窗口探针**：`window_size == 0` 时按 **1** 处理（`_window_size = window_size == 0 ? 1 : window_size`），硬挤出一个 1 字节探测段，**诱使接收方重新通告窗口**。否则双方死锁（发送方永远不知道窗口何时重开）。
- **空段（纯 ACK）**：收到携带数据的段但自己无数据可发时，回 `send_empty_segment()`（不携带数据、只设 seqno/ackno）。所以"触发发送"有时发的是**空段**而非数据段。
- **定时器语义**：`ack_received()` 里**部分**确认 → 重启 RTO、**全部**确认 → 停表；`tick()` 负责超时重传，并在接收方仍有窗口时做 **RTO 退避（翻倍）**。

### ⑤ 与真实 TCP 的差别（重要，别把 lab 当现实）

| 维度 | CS144 lab | 真实 TCP（Linux） |
|---|---|---|
| **拥塞控制** | **完全不实现**——窗口只有 **rwnd**（接收方通告），没有 cwnd | `cwnd` + `rwnd` 取 min |
| **发送节奏** | 无 pacing，一次 `fill_window()` 可把整窗一口气推出（**burst**） | cwnd 限制 + pacing 平滑 |
| 其它 | 无 TSO/GSO、无 Nagle、无延迟 ACK | 全都有 |

- 结论：**CS144 的 lab 是「流控 + 可靠传输 + 连接管理」的模型，不含拥塞控制**（讲义讲，lab 不做）。
- 所以"ACK 一到、窗口怎么开就怎么发"在 lab 成立；真实网络还要再被 **cwnd** 卡一道。

### ⑥ 本地仓状态（核实）

- `learning-408/cs144` 当前在 **checkpoint 1**（分支 `check1-startercode`，commit `4b75047`），`src/` 只有 `byte_stream.*` + `reassembler.*`，**尚无 `tcp_sender` / `tcp_connection`**——以上属 lab3 / lab4 内容。

### ⑦ 易错点

- ❌ 「每个 ACK → 发一个新包」：触发的是 `fill_window()` 的**循环**，数量由窗口剩余量决定。
- ❌ 把 CS144 的窗口当成 cwnd：lab 里**没有拥塞控制**，只有 rwnd。
- ❌ 忘了零窗口要按 1 处理：会导致连接死锁，测不过。


## 传输层 · 块3 前置补讲：SYN / FIN 各消耗 1 个序号（序号消耗规则）

> 2026-10-12 · 来源：学习对话（块3 前置，补齐块2 未讲透的点）

### ① 一句话规则

一个 TCP 报文段消耗的序号数 = **SYN(1) + FIN(1) + 数据字节数**；**纯 ACK 段消耗 0**。

通用式：**下一段 seq = 本段 seq + 本段消耗的序号数**

### ② 四种情况

| 段类型 | 消耗序号 | 例子 |
|---|---|---|
| 数据段 | len(数据) | seq = 100，发 200 B → 下一段 seq = 300 |
| **SYN 段** | **1** | seq = x → 对端下次期望 **x + 1** |
| **FIN 段** | **1** | seq = u → 对端下次期望 **u + 1** |
| **纯 ACK 段** | **0** | 不占序号，也不会被确认 |

- **CS144 对应 API**：`TCPSegment::length_in_sequence_space()` —— 它正是"本段消耗多少序号"。**它不等于 payload 长度**：SYN / FIN 段 payload = 0，但它返回 **1**。

### ③ 为什么 SYN / FIN 必须占一个序号

因为它们**必须被可靠确认**。若不占序号，对端就无法用 `ack` 表达"我收到了你的 SYN / FIN"，超时重传也无从判断该重传哪一段。占 1 个序号是"把它们当作 1 字节数据来确认"的最小代价方案。

### ④ 直接后果（块3 会用到）

- 三次握手完成后，**主动方第一个数据字节的序号 = ISN + 1**（ISN 被自己的 SYN 吃掉）；被动方同理。
- 四次挥手时，**FIN 之后的下一个序号 = 本段 seq + 1**。
- **最后的 ACK 不占序号** → 它丢了**没有重传机制兜底** → 这正是 **TIME_WAIT 的动机之一**（主动方等 2MSL，让被动方能重传 FIN、自己再补一个 ACK）。

### ⑤ 与已学内容的接口

- 上游（块2）：面向字节流 → 序号是**字节编号**，不是"第几个报文段"。
- 下游（块3）：三次握手 / 四次挥手的 `seq` `ack` 计算全部基于本规则。
- 旁证（CS144）：`length_in_sequence_space()` 就是这条规则在代码里的名字。

### ⑥ 易错点

- ❌ 以为"控制段不占序号"：SYN / FIN 各占 **1** 个。
- ❌ 把 `length_in_sequence_space()` 当成 payload 长度：SYN 段 payload = 0，但它返回 1。
- ❌ 算握手 / 挥手的 ack 时忘了 **+1**。


## 传输层 · 块3 TCP 连接管理（三次握手 / 四次挥手 / TIME_WAIT / 状态机）★核心

> **内核一句话**：TCP 的「连接」不是网络里的一根线，而是**双方各自维护的一份状态**。建立 = 双方各自建立并确认；释放 = 双方各自销毁。因为「各自」，所以才有来回。

**必考点清单**
1. 三次握手的报文序列 + seq / ack 计算 ★核心
2. 为什么是三次，不是两次也不是四次 ★核心
3. 四次挥手 + 为什么是四次（半关闭）★核心
4. TIME_WAIT 与 2MSL 的两个作用 ★核心
5. 11 种状态与迁移 ☆常规
6. SYN 洪泛 / SYN cookie ☆了解

---

### 3.1 ① 机制层 —— 三次握手

```
A（客户端）                          B（服务端）
CLOSED                              LISTEN
   |--- SYN=1, seq=x ------------------->|         A: SYN-SENT
   |<-- SYN=1, ACK=1, seq=y, ack=x+1 ----|         B: SYN-RCVD
   |--- ACK=1, seq=x+1, ack=y+1 -------->|         A: ESTABLISHED
ESTABLISHED                          ESTABLISHED
```

- **第 1 条 `SYN=1, seq=x`**：SYN 表示「要建连、请同步序号」；`x` 是 A 的**初始序号 ISN**（随机取，不从 0 开始）。不携带数据，**消耗 1 个序号**。
- **第 2 条 `SYN=1, ACK=1, seq=y, ack=x+1`**：一个报文干两件事——ACK 掉对方的 SYN（`ack=x+1`，**+1 就是给 SYN 吃掉的序号**）+ 同步自己的 ISN（`seq=y`）。**两次握手能省，正是因为服务端这两件事可以合并。**
- **第 3 条 `ACK=1, seq=x+1, ack=y+1`**：确认 B 的 SYN；`seq=x+1` 是 A 第一个数据字节的编号（x 已被自己的 SYN 占用）。**不携带数据、不消耗序号（纯 ACK）。**

**序号规则（锁死全部计算）**
> 数据段 seq = 本段第一个数据字节的编号；**SYN / FIN 各占 1 个序号**；**纯 ACK 占 0**。

**状态迁移（注意不对称）**

| 时刻 | A（客户端） | B（服务端） |
|---|---|---|
| 起始 | CLOSED | LISTEN |
| 发出 SYN 后 | SYN-SENT | LISTEN |
| 收到第 2 条后 | **ESTABLISHED** | SYN-RCVD |
| 收到第 3 条后 | ESTABLISHED | **ESTABLISHED** |

A 在第 2 条到达时即可发数据；B 要等第 3 条。所以第 3 条的性质，对 A 是「确认对方也准备好了」，对 B 才是「完成」。

---

### 3.2 ① 机制层 —— 四次挥手

```
A（主动关闭）                        B（被动关闭）
ESTABLISHED                          ESTABLISHED
   |--- FIN=1, seq=u ------------------->|         A: FIN-WAIT-1
   |<-- ACK=1, seq=v, ack=u+1 -----------|         B: CLOSE-WAIT / A: FIN-WAIT-2
   |      （B 可继续发数据：半关闭窗口）  |
   |<-- FIN=1, ACK=1, seq=w, ack=u+1 ----|         B: LAST-ACK
   |--- ACK=1, seq=u+1, ack=w+1 -------->|         A: TIME-WAIT(2MSL) / B: CLOSED
```

**为什么挥手是四次、握手只要三次**：TCP **全双工，两个方向各自独立关闭**，核心概念 **半关闭（half-close）**。A 发 FIN 只说「A→B 方向我没数据了」，**B→A 方向照常工作**。于是第 2、3 条之间夹着一个窗口——**B 可能还有响应数据没发完**：先回 ACK，继续发完，再发自己的 FIN。握手时服务端能把 ACK 和自己的 SYN 合并（第 2 条），挥手时**不能合并**，因为对 B 来说「收到你的 FIN」与「我也发完了」是**两个不同时间点**。

**逐条**

| # | 方向 | 报文 | 含义 |
|---|---|---|---|
| 1 | A→B | `FIN=1, seq=u` | 我发完了 |
| 2 | B→A | `ACK=1, seq=v, ack=u+1` | 知道了（FIN 也占 1 个序号） |
| 3 | B→A | `FIN=1, ACK=1, seq=w, ack=u+1` | 我也发完了 |
| 4 | A→B | `ACK=1, seq=u+1, ack=w+1` | 知道了 |

第 2、3 条的 `ack` **都是 u+1**：中间 B 只发数据、没收到 A 的新数据，确认号不变。

---

### 3.3 TIME_WAIT 与 2MSL：两个不可替代的作用

**MSL = Maximum Segment Lifetime**（最长报文段寿命），一个 TCP 报文能在网络中存活的最长时间（工程常取 30s / 1min / 2min）。**2MSL = 一个「一来一回」的上界。**

**作用一：保证 B 的最后一条 FIN 能收到 ACK（防止对方关不掉）**
第 4 条 ACK **丢了不重传**（不携带数据、不消耗序号，TCP 不为它重传）。若 A 发完立刻消失，B 会卡在 LAST-ACK 超时后**重发 FIN**，而 A 已不存在 → **B 永远关不掉**。所以 A 发完 ACK 后**留在 TIME-WAIT 等 2MSL**：期内若 B 重传 FIN，A 还能重发 ACK（「FIN 重传 + 重传的 ACK」一个来回 < 2MSL，足够）；2MSL 内没等到，说明 ACK 已送达。

**作用二：让本次连接的「迷途报文」全部消散（防止污染新连接）**
连接关闭后同一四元组可能**立刻被新连接复用**；若老连接的延迟报文还在网络里游荡，可能被新连接当成自己的数据收下 → **数据串了**。等 2MSL 保证本次连接产生的所有报文都已消失（任何报文最多活 MSL）。

**高频副作用**：若**服务端主动关闭**，就进入 TIME-WAIT 并占用端口 2MSL。高并发短连接场景下**大量 TIME-WAIT 会耗尽服务端可用端口** —— 这就是「服务端不宜主动关闭连接」的由来。

**TIME-WAIT 与 CLOSE-WAIT 别混**

| 状态 | 在哪一方 | 含义 | 大量出现说明 |
|---|---|---|---|
| **CLOSE-WAIT** | **被动**关闭方 | 收到对方 FIN，**自己数据还没发完**，先回 ACK 挂着 | 应用没及时 `close()`（代码问题） |
| **TIME-WAIT** | **主动**关闭方 | 已发完最后 ACK，**等 2MSL 保对方收到** | 主动关闭太频繁（架构问题） |

---

### 3.4 11 种状态的全貌

**两条主链**
- **主动方**：CLOSED → SYN-SENT → ESTABLISHED → FIN-WAIT-1 → FIN-WAIT-2 → **TIME-WAIT** → CLOSED
- **被动方**：CLOSED → LISTEN → SYN-RCVD → ESTABLISHED → CLOSE-WAIT → LAST-ACK → CLOSED

**只有 CLOSED 与 LISTEN 是「静止」状态**（可长期停留）；其余都是过路。整条链上**唯一带「等待」语义的是 TIME-WAIT** —— 所以它是考点。

**两个特殊状态（408 会考）**
- **CLOSING**：**双方同时**关闭时出现（自己发了 FIN 还没等到 ACK，就先收到对方的 FIN）→ 双方都要等各自的 ACK
- **同时打开**（罕见，双方都主动 SYN）：SYN-SENT → SYN-RCVD → ESTABLISHED

---

### 3.5 ② 设计层 —— 四个「为什么」

**① 为什么是三次不是两次？**
两次的话 B **无法确认「A 收到了我的 SYN+ACK」**。具体挂在**失效连接的延迟 SYN**：一个早已废弃的连接请求在网络里滞留、某刻才到达 B，B 误以为是新请求，回 ACK、分配资源、建立连接。若只有两次握手，该连接**白白建立**而 A 不知情，**B 的资源被浪费**（SYN 洪泛攻击的理论基础）。有第三次，A 能回 RST 否决。**第三次的本质 = 给客户端一个否决机会。**

**② 为什么 ISN 要随机取？**
若 ISN 可预测，攻击者可**伪造**出对方期望序号的报文，不经握手直接注入数据（半开连接攻击）。随机化 ISN 让序号无法猜中。

**③ 为什么结束要等 2MSL？**
见 3.3。要点：**主动方多承担 2MSL 代价，换「对方能关掉」和「新连接不被污染」**。

**④ 为什么连接要「每方各自维护」？**
回到内核那句话：连接是双方各自的一份状态，不是网络里的一根线。推论：① 一方崩了另一方可能还「以为连着呢」（靠 keepalive / 超时发现）；② 状态可以**不对称**（一端 CLOSE-WAIT、另一端 FIN-WAIT-2）；③ **没有中心管理者** —— 路由器和链路层完全不知道 TCP 连接这回事（这正是「网络层无连接、传输层有连接」能共存的原因）。

---

### 3.6 ③ 验收层

| # | 题型 | 模板要点 | 优先级 |
|---|---|---|---|
| 1 | **序号 / 确认号计算** | 数据段 seq = 首字节编号；SYN/FIN 各占 1；纯 ACK 占 0 | ★核心 |
| 2 | **握手过程与排序** | 三段名、标志位、状态迁移三样齐全 | ★核心 |
| 3 | **为什么不是两次 / 为什么是四次** | 「防失效连接请求」/「半关闭，服务端可能还有数据」 | ★核心 |
| 4 | **TIME-WAIT 两个作用** | 保最后 ACK 送达 + 让迷途报文消散 | ★核心 |
| 5 | 状态识别 | 给场景问状态（CLOSE-WAIT / TIME-WAIT 最常考） | ☆常规 |
| 6 | SYN 洪泛与防护 | SYN cookie：不预先分配资源，先算 cookie | ☆了解 |

**代表例题**：A 主动连接 B，A 的 ISN = 1000，B 的 ISN = 5000。建连后 A 向 B 发 200 字节，B 回复 100 字节，随后 A 主动关闭。
（1）三次握手三条报文的 seq / ack？
（2）A 数据段的 seq、B 的确认号？
（3）挥手四条报文的 seq / ack？A 最终进什么状态、停多久？

**解**
（1）

| # | 报文 | seq | ack | 标志 |
|---|---|---|---|---|
| 1 | A→B | 1000 | — | SYN |
| 2 | B→A | 5000 | 1001 | SYN, ACK |
| 3 | A→B | 1001 | 5001 | ACK |

（2）A 第一个数据字节 = **1001**（1000 被自己的 SYN 吃了），200 字节占 1001~1200 → B 的确认号 = **1201**。B 数据从 5001 到 5100。

（3）

| # | 报文 | seq | ack |
|---|---|---|---|
| 1 | A→B FIN | 1201 | — |
| 2 | B→A ACK | 5101 | 1202 |
| 3 | B→A FIN | 5101 | 1202 |
| 4 | A→B ACK | 1202 | 5102 |

**A 进入 TIME-WAIT，停留 2MSL**，之后才回 CLOSED。第 3 条 FIN 的 seq 仍是 **5101**：B 那条纯 ACK 不消耗序号，B 的「下一个待发字节」仍是 5101，FIN 从这个编号发（FIN 自己占掉 5101）。

**易错点**
- ❌ 忘记 SYN / FIN 各占 1 个序号 → **所有 ack 全体少 1**
- ❌ 数据段 seq 与上一段相同 → 下一段 seq = 本段 seq + 本段数据长度（+ SYN/FIN 的 1）
- ❌ 「挥手也是三次」→ **四次**，多的那条来自**半关闭**
- ❌ 把 TIME-WAIT 判给被动方 → **主动关闭方**才进 TIME-WAIT
- ❌ 以为最后那条 ACK 丢了会重传 → **不重传**，靠主动方等 2MSL 兜底

---

**408 边界**：只考三次握手 / 四次挥手 / 状态机 / TIME-WAIT 的纸面推演与「为什么」；**TCP option 协商细节（MSS / 窗口扩大 / SACK）、SYN cookie 的具体算法、tcp_tw_reuse 等内核调参不考**。


## 传输层 · 块3 补讲：第 3 条 ACK 的 seq 与首个数据段 seq 相同（纯 ACK 不推进序号）

> 问题：三次握手第 3 条是 `ACK=1, seq=x+1, ack=y+1`，那客户端接下来发数据，seq 还是 x+1 吗？
> **答：是，仍是 x+1。**

**原因**：`seq` = 「本报文段**第一个字节**在字节流中的位置」。第 3 条 ACK **不携带数据**，它只是**指向** x+1 这个位置说「我下一个要发的是 x+1」，但**没有真正占用**它。所以下一个数据段还是从 x+1 起步。

**核心模型（比背规则有用）**
> **`seq` 指针只在「真正消耗序号空间」时前移。** 会消耗的只有三样：**数据字节、SYN、FIN**；**纯 ACK 一律不消耗**。

因此「第 3 条 ACK 的 seq」与「第一个数据段的 seq」是**同一个数** —— 它们指向同一个位置。

**若第 3 条 ACK 顺便携带数据**（TCP 允许，实现中少见），设带 n 字节：

| | seq | 消耗 | 下一个待发字节 |
|---|---|---|---|
| 第 3 条不带数据（常见） | x+1 | 0 | **x+1** |
| 第 3 条带 n 字节 | x+1 | n | **x+n+1** |

**与代表例题自洽**：例题中 A 的 ISN = 1000，第 3 条 ACK 为 `seq=1001`，紧接着 A 发的 200 字节数据段 **seq 也是 1001** —— 两者相同，正好印证。

**顺带钉死一个结构：第 3 条 ACK 里有两个独立计数器**

| 字段 | 含义 | 数值 | 谁维护 |
|---|---|---|---|
| `seq` | **我**发到哪了（我下一个要发的字节编号） | x+1 | A 的发送序号 |
| `ack` | 我**期待你**从哪发（你下一个该发的字节编号） | y+1 | A 对 B 的接收序号 |

读出来是两句独立的话：「我下一步从 x+1 开始发」+「请你从 y+1 开始发」。**`seq` 与 `ack` 是两条完全独立的计数器，永远不要连起来算。**

**直观类比**：第三次握手那条 ACK 像一次「报到」—— 它登记了「我准备好了、编号从 x+1 开始」，但**没有领走任何一个编号**；所以正式发数据时，第一个编号还是 x+1。


## 传输层 · 块4 可靠传输与流量控制（滑动窗口 / GBN / SR / RTO）★核心

> 与前后块的分工：块2 讲首部（序号在哪）、块3 讲连接（怎么开怎么关）、块5 讲拥塞（网络能承受多少）；**块4 讲夹在中间的那件事：怎么把字节流一个不丢、一个不乱地送过去。**

**必考点清单**
1. 停等 / GBN / SR 三种协议的行为与窗口规则 ★核心
2. **GBN 窗口 ≤ 2ⁿ−1，SR 窗口 ≤ 2ⁿ⁻¹** ★核心
3. 信道利用率计算 ★核心
4. TCP 发送窗口 = min(rwnd, cwnd) − 已发未确认 ★核心
5. RTO 计算与 Karn 算法 ☆常规
6. 流量控制：零窗口 / 持续计时器 / 糊涂窗口综合症 ☆常规

---

### 4.1 ① 机制层

#### 4.1.1 停等协议（起点）

发一帧、等确认、再发下一帧。致命问题：**信道大部分时间是空的**。

```
U = T_D / (T_D + RTT + T_A)
```

例：1 Mbps 链路，1000 bit 帧，RTT = 20 ms → T_D = 1 ms，U ≈ 1/(1+20) ≈ **4.8%**。**96% 的带宽在等确认**——因为 T_D 是推比特上线的时间，RTT 由距离决定，改不了。

#### 4.1.2 滑动窗口 = 让多个帧同时在途

**发送窗口 W = 允许「已发出但未确认」的帧数上限**（就是前面反复出现的「在途量上限」）。

```
U = W · T_D / (T_D + RTT + T_A)      （W·T_D ≥ T_D + RTT + T_A 时 U 取 1）
```

#### 4.1.3 GBN（后退 N 帧）vs SR（选择重传）

分歧只在**接收方怎么处理乱序帧**：

| | **GBN** | **SR** |
|---|---|---|
| 接收窗口 | **1**（只收按序的） | **> 1**（缓存乱序帧） |
| 收到乱序帧 | **丢弃** + 重发最后正确帧的 ACK | **缓存** + 单独 ACK 该帧 |
| 确认方式 | **累积确认**（ACK n = n 及之前全收到） | **逐帧确认** |
| 出错时重传 | **从出错帧起，后面全部重传** | **只重传那一帧** |
| 发送窗口上限 | **W ≤ 2ⁿ − 1** | **W ≤ 2ⁿ⁻¹** |

**出错行为对比（发 1~5，第 2 帧丢了）**
- GBN：接收方**丢弃** 3/4/5（不按序），超时后**连带全部重传** 2,3,4,5
- SR：接收方**缓存** 3/4/5 并各自 ACK，超时后**只重传 2**

#### 4.1.4 两个窗口上限的推导（最易记混，理解了不用背）

核心矛盾：**序号空间只有 2ⁿ 个，发送方要循环使用**。窗口开太大 → 新旧两轮序号**重合** → 接收方分不清「新帧还是重传的旧帧」。

- **GBN：W ≤ 2ⁿ − 1**。接收窗口 = 1，发送窗口只要**不占满序号空间**、留 1 个号当分界即可。
- **SR：W ≤ 2ⁿ⁻¹**。接收方还会**缓存乱序帧**（手里攥着一批序号），约束更强：**发送窗口 + 接收窗口 ≤ 2ⁿ**。取发送窗口 = 接收窗口 = W，即 `2W ≤ 2ⁿ` → `W ≤ 2ⁿ⁻¹`。

**n = 3（序号 0~7）的直观验证**
- W = 8 = 2ⁿ：第 1 轮窗口 0~7、第 2 轮窗口 0~7 —— **完全重合**，收到「0」分不清是新帧还是重传 → 重复交付
- W = 7 = 2ⁿ−1：第 1 轮 0~6、第 2 轮 7,0,1,2,3,4,5 —— **起点不同（0 vs 7）**，收到「0」能判定是重传

#### 4.1.5 TCP 的发送窗口（字节视角）

发送缓冲区分四段，窗口覆盖中间两段：

```
 已发送已确认 | 已发送未确认 | 可发送未发送 | 不可发送（窗口外）
      ↑                ↑             ↑
  （可丢弃）    （= 在途量）  （窗口开着待发）
                       └──── 发送窗口 = min(rwnd, cwnd) ────┘
```

- 任意时刻：**已发送未确认 ≤ min(rwnd, cwnd)**
- ACK 回来 → 左边界右移 → 窗口滑动 → 后面的字节进入可发区 → 立刻补发（这正是「水位驱动」）
- 两个指针：**左边界 = 最早未确认**、**下一个要发的位置**

#### 4.1.6 TCP 的可靠传输：GBN 与 SR 的混血

| 特征 | TCP 的做法 | 像谁 |
|---|---|---|
| 累积确认（ack = 期望的下一个字节） | ✓ 标准行为 | **GBN** |
| 收到乱序段 | 缓存（接收缓冲区等空洞补齐） | **SR** |
| 一个段丢了 | 3 个重复 ACK → **快重传**只重传那一段 | **SR** |
| 超时 | 只重传最早的未确认段 | GBN 的变体 |

**SACK（选择性确认）**：接收方在选项里告知「我还收到了 5000~6000、7000~8000」，发送方就知道只有 6000~7000 缺，不必退回去重传一大片 —— **把 SR 的优点嫁接进累积确认**。

#### 4.1.7 超时重传时间 RTO

```
RTT_S ← (1 − α)·RTT_S + α·新样本            加权平均 RTT，α = 1/8
RTT_D ← (1 − β)·RTT_D + β·|新样本 − RTT_S|   偏差，β = 1/4
RTO = RTT_S + 4·RTT_D
```

读法：RTT_S = "平均往返时间"的滑动平均（新样本权重 1/8）；RTT_D = "波动幅度"的滑动平均（新样本权重 1/4）。**RTO = 平均值 + 4 倍波动**，网络越抖放得越宽。

**Karn 算法**：**重传过的报文段，其 ACK 不用于更新 RTT 样本** —— 超时重传后收到的 ACK 分不清是「原来那次」还是「重传那次」的确认，用它算 RTT 会偏小。

**退避**：每超时一次 RTO **翻倍**（指数退避），避免拥塞时反复砸包。

#### 4.1.8 流量控制

**流量控制 = 别冲爆接收方的缓冲区**（保护**接收方**，不是网络）。手段是接收方在首部通告 **rwnd**。

- **零窗口**：缓冲区满，通告 rwnd = 0，发送方停下。若之后接收方腾出空间、新窗口通告**丢了** → **死锁**（发送方等窗口、接收方等数据）。
  - 解法：**持续计时器（坚持计时器）** —— rwnd = 0 时启动，到期发一个 **1 字节探测段**，逼接收方重新通告窗口。（课本出处：CS144 里「零窗口按 1 处理」）
- **糊涂窗口综合症（SWS）**：接收方每次只腾一点空间就通告，或发送方每次只发很小的段 → **每个小段要 40 字节首部**，开销比数据还大。
  - 解法：接收方等**空间够半个缓冲区或一个 MSS** 再通告（Clark 算法）；发送方用 **Nagle 算法**（攒够一个 MSS 或等到 ACK 再发）。

---

### 4.2 ② 设计层 —— 三个「为什么」

**① 为什么要有窗口（不用停等）？**
**传播时延是物理定律，改不了**。停等协议下利用率被 `T_D/(T_D+RTT)` 锁死，链路带宽再高也没用。窗口的本质是**用「允许在途的量」去填满 带宽 × RTT 这个管道**（带宽时延积）。**窗口太小 = 管道饿着；窗口够大 = 链路跑满。**

**② 为什么 GBN 和 SR 的窗口上限不一样？**
同一句话概括：**窗口越大越容易把序号空间用「绕圈」**（新旧两轮序号重叠），而接收方缓存越多，越容易撞上。
- GBN 接收窗口 = 1 → 发送窗口可占 `2ⁿ − 1`（留 1 个号当分界）
- SR 接收窗口 = W → **发送 + 接收 ≤ 2ⁿ** → `W ≤ 2ⁿ⁻¹`

**③ 为什么 TCP 用累积确认？**
因为它**抗 ACK 丢失**：累积确认下，ACK 丢失会被后续 ACK 自动覆盖；逐帧确认里丢一个 ACK 就要重传一个已正确到达的帧，纯浪费。代价是「一个洞要重传一片」，所以又用 **SACK** 和**快重传**来补 —— **典型的「先拿鲁棒性，再用补丁换效率」**。

---

### 4.3 ③ 验收层

| # | 题型 | 模板要点 | 优先级 |
|---|---|---|---|
| 1 | **窗口上限** | GBN → `2ⁿ − 1`；SR → `2ⁿ⁻¹`。给序号位数直接套 | ★核心 |
| 2 | **信道利用率** | `U = W·T_D / (T_D + RTT + T_A)`；停等时 W = 1 | ★核心 |
| 3 | **滑动窗口推演** | 列「已确认 / 已发未确认 / 可发 / 不可发」，逐时刻推进 | ★核心 |
| 4 | **TCP 有效窗口** | `min(rwnd, cwnd) − 已发未确认` | ★核心 |
| 5 | RTO / Karn | 加权重算 + 「重传段不采样」 | ☆常规 |
| 6 | 零窗口与 SWS | 持续计时器 / Nagle / Clark | ☆常规 |

**代表例题**
> A、B 用滑动窗口协议通信，**序号字段 4 位**。
> （1）GBN 的发送窗口最大是多少？
> （2）SR 的发送窗口最大是多少？
> （3）链路 8 Mbps，单向传播时延 15 ms，帧长 2000 bit，确认帧长度忽略。停等协议下信道利用率是多少？若用 GBN 达到 100% 利用率，窗口至少多少？

**解**
（1）序号 4 位 → 2⁴ = 16 个序号 → GBN：**W ≤ 16 − 1 = 15**
（2）SR：**W ≤ 16 / 2 = 8**
（3）
- T_D = 2000 / 8×10⁶ = **0.25 ms**；RTT = 2 × 15 = **30 ms**
- 停等：`U = 0.25 / (0.25 + 30) ≈ 0.83%`
- 要 100%：`W·T_D ≥ T_D + RTT` → `W ≥ (0.25 + 30) / 0.25 = 121` → **W ≥ 121**

**（3）是考点**：算出的 121 远大于（1）的 15 —— **4 位序号根本不够用**。窗口受「序号位数」与「利用率需求」**双重约束，取小**。

**易错点**
- ❌ 两个上限记反（GBN 记成 2ⁿ⁻¹、SR 记成 2ⁿ−1）—— **GBN 松弛（−1）、SR 严格（减半）**
- ❌ 利用率分母漏 T_A，或把 RTT 写成单向时延
- ❌ 把 SR 的重传理解成「重传全部未确认」—— **那是 GBN**
- ❌ 有效窗口忘了减「已发未确认」—— `min(rwnd, cwnd)` 是**上限**，不是「现在还能发多少」
- ❌ 零窗口时发数据而不是探测段 —— **零窗口只能发 1 字节探测**

**408 边界**：只考停等 / GBN / SR 的窗口规则与利用率计算、TCP 发送窗口与累积确认、RTO 的基本公式与 Karn 算法；**SACK 的选项格式、Nagle / Clark 的实现细节、tcp_tw_* 等内核调参不考**。


## 传输层 · 块4 补讲：发送窗口落在哪一级 —— 三级存储与「留档到 ACK」

> 承接块4「TCP 发送窗口」与前面「内核实现 / 网卡 IO」几讲：**发送窗口和网卡发包不是一处，是串联的两级。**

**三级存储（数据从应用到网线）**

```
应用进程 --write()--> ② socket 发送缓冲区 --> ③ qdisc 队列 --> ④ 网卡 TX ring --DMA--> 网线
```

| 位置 | 谁在管 | 容量约束 | 数据何时消失 |
|---|---|---|---|
| ② socket 发送缓冲区 | **TCP** | cwnd / rwnd | **ACK 确认之后**才释放 |
| ③ qdisc 队列 | 内核流量整形 | 队列长度（包数） | 提出去就没了 |
| ④ 网卡 TX ring | **网卡硬件** | 描述符格数（常见 256~4096） | **DMA 读走后立即释放** |

**三个要点**

1. **窗口本身不是一块内存**。数据存在发送缓冲区里；窗口是 TCP 控制块里的几个整数（最早未确认 / 下一个要发 / 对端通告窗口）+ 作用在缓冲区字节序列上的**区间语义**。准确说法：**缓冲区是仓库，窗口是账本，ring 是装车口。**
2. **两个约束是「与」的关系**，不是「或」：能发 = `在途量 < min(cwnd, rwnd)` **且** `ring 还有空描述符`。哪个先卡住就在这里停下。
3. **最本质的差别是「留档时机」**：发送缓冲区**留档到 ACK**（因为丢了必须重传）；ring **发出即弃**（网卡 DMA 读走就腾位置，**网卡没有任何重传能力**）。

**这条差别正好接上链路层那条结论**：以太网「只检错、不重传」，所以包在链路上丢了链路层根本不补 —— **能补的只有握着副本的 TCP**。发送缓冲区那份「留档」就是重传能力的前提；网卡 ring 里没有留档，注定管不了可靠性。

**追问：ring 满了怎么办？** TCP 停下，等网卡发完中断（TX completion）回收描述符再继续 —— 这就是「事件驱动」的**第四个触发源**（应用写入 / ACK 到达 / 超时 / 网卡腾出描述符）。

**为什么不合成一个？** 它们防的是两个不同的瓶颈：**窗口怕压垮网络**（路由器和链路），**ring 怕压垮网卡硬件**。瓶颈独立，约束就必须独立。

**408 边界**：qdisc / TX ring / BQL 都是实现层，考纲不涉及；这里只为锚定「窗口 = 逻辑额度 ≠ 硬件队列」这个直觉。



## 传输层 · 块4 补讲：GBN「留一个序号当分界」到底留的是什么

**先纠一个数**：序号空间是 **0 ~ 2ⁿ−1**，共 2ⁿ 个。**没有「2ⁿ」这个序号** —— 2ⁿ 在模 2ⁿ 运算下就是 0。

**统一式（一句话记死）**

> **发送窗口 + 接收窗口 ≤ 2ⁿ**

- **GBN**：接收窗口 = 1 → `W + 1 ≤ 2ⁿ` → **W ≤ 2ⁿ−1**。那个「1」就是留出来的那一格，**它留给接收窗口占位**。
- **SR**：接收窗口 = W → `2W ≤ 2ⁿ` → **W ≤ 2ⁿ⁻¹**。

**「分界」分的是什么：上一轮的帧 vs 这一轮的帧**

反证（n = 3，序号 0~7）：

| 窗口大小 | 发送窗口 | 接收窗口位置 | 结果 |
|---|---|---|---|
| **W = 8 = 2ⁿ** | 0~7（占满） | 8 mod 8 = **0** | **与发送窗口左端重合** |
| **W = 7 = 2ⁿ−1** | 0~6 | 7 | 错开一格，可分 |

- **W = 8 时**：发送方连发 0~7 全在途，正好把序号空间用满；接收方期待的下一帧 = 8 ≡ 0，**和发送窗口最早的 0 撞在同一个位置**。若 ACK 全丢、发送方超时**重传帧 0**，接收方看到「0」**分不清这是新一轮的新帧还是上一轮的重传** → 当作新帧收下 → **重复交付**。
- **W = 7 时**：第 1 轮窗口 0~6，第 2 轮窗口 7,0,1,...。接收方期待 7，重传的却是 0 —— **序号对不上**，一眼判定是重传，安全丢弃。

所以那个「空出来的序号」是**代际标记**：**发送窗口左端与接收窗口当前位置之间必须差至少一格**，才能区分两轮。

**⚠️ 一个数值巧合陷阱（易混的根源）**

- 「**最大序号** = 2ⁿ−1」← 来自「序号从 0 开始编号，共 2ⁿ 个」
- 「**GBN 窗口上限** = 2ⁿ−1」← 来自「留 1 格给接收窗口」

**两个数一样，来路完全不同。** 看到 2ⁿ−1 要先问自己：这是「序号最大值」还是「窗口能装几帧」。

| 量 | 数值 | 怎么来的 |
|---|---|---|
| 序号个数 | 2ⁿ | 0 ~ 2ⁿ−1 |
| 最大序号 | 2ⁿ−1 | 编号从 0 起 |
| GBN 发送窗口上限（帧数） | 2ⁿ−1 | 2ⁿ − 接收窗口(1) |
| SR 发送窗口上限（帧数） | 2ⁿ⁻¹ | 2ⁿ − 接收窗口(W) |

**统一记法：窗口上限 = 2ⁿ − 接收窗口大小。**



## 传输层 · 块4 补讲：TCP 的「字节序号账本」，与重传到底做了什么

### 一、TCP 管的是「字节序号账本」，不是「包的索引」

一句话：**哪些能发、该发哪个、收到哪些、缺哪个** —— 这四件事就是 TCP 的全部工作。但它管的是**字节的位置**，不是「第几个包」：TCP 面向字节流，**根本不认识「包」**（报文段只是临时的打包动作）。

**四个问题 ↔ 四个量（Linux 命名）**

| 问题 | 内核里的量 | 含义 |
|---|---|---|
| 收到哪些了 | `snd_una` | 最早一个还没被确认的字节 |
| 该发哪个了 | `snd_nxt` | 下一个要发出去的字节 |
| 哪些能发 | `snd_wnd` / `snd_cwnd` | 对端通告窗口 / 拥塞窗口，**取小** |
| 缺哪个 | `rcv_nxt` | 我期待你发来的下一个字节 |
| （告诉对方我能收多少） | `rcv_wnd` | 我的接收窗口 |

**「现在能发哪些字节」的完整答案**

```
能发的区间 = [ snd_nxt , snd_una + min(snd_wnd, snd_cwnd) )
             └该发哪个┘   └──── 收到哪些 + 窗口允许多少 ────┘
```

**结论**：TCP 连接的「状态」，本质就是**这几个整数的值**。缓冲区里存的是数据，而这几个整数决定**数据的哪一段算数**。「账本索引」这个理解正确 —— 只是索引的单位是**字节**，不是包。

### 二、重传 = 把内存里的字节「再投递一次」

**是的，物理上就是再塞一次。** 靠的正是「发送缓冲区留档到 ACK」——ring 里的副本早被 DMA 读走释放，**只有缓冲区那份还能再取一次**。

**三件事说清**

1. **从哪取**：从发送缓冲区（那份留档）。
2. **取出来的是什么**：**同一批字节 + 全新的首部**。
   - **seq 不变**（还是那段字节）—— 接收方正是靠这个认出「这是重传」
   - **但首部是重造的**：IP 首部、以太网帧头、IP 标识都是新的
   - 准确说法：**拿同一份数据重新打包一次**，不是「原封不动再发一遍」
3. **走的路径**：与首传**完全一样**，没有任何特殊待遇（qdisc → ring → 网卡）。

**⚠️ 由此得到的后果**：**重传会加剧拥塞** —— 网络一堵 → 丢包 → 重传 → 更堵 → 更多丢包。**这就是 RTO 必须指数退避的根本原因。**

**重传粒度**（与块4 的 GBN/SR 对上）

| 触发 | 重传范围 | 像谁 |
|---|---|---|
| 超时（RTO） | 只重传**最早那个未确认段** | GBN 变体 |
| 3 个重复 ACK | 只重传**被判定丢失的那一段** | SR |
| 开了 SACK | 精确补**缺的那几块** | SR（增强） |

### 三、接收方拿同一本账做的三个判断

收到任何一段，第一件事是拿 `seq` 和 `rcv_nxt` 比：

| 比较 | 判定 | 动作 |
|---|---|---|
| `seq = rcv_nxt` | 正常 | 收下，前移 `rcv_nxt`，回 ACK |
| `seq > rcv_nxt` | 中间**有洞** | 缓存（TCP 式）或丢弃（GBN 式），回**重复 ACK** |
| `seq < rcv_nxt` | **重复段**（含重传包） | 丢弃，**重发 ACK**（「我从 rcv_nxt 开始要」） |

**重传包落到第三行**：收下也没用，回一个 ACK 就够。**真正有用的不是重传的包，而是它触发的那次 ACK。**

由此补上「序号管理」的另一半：**同一个 seq 既能发现「缺了」，也能发现「重复了」。**

| 谁 | 拿什么比 | 结论 |
|---|---|---|
| 发送方 | 收到的 `ack` vs `snd_una` | 哪些被确认 → 哪些要重传 |
| 接收方 | 收到的 `seq` vs `rcv_nxt` | 新的 / 有洞 / 重复 → 怎么回 |

### 四、一个反直觉结论：重传有可能白做

若原包没丢、只是**它的 ACK 丢了**，发送方超时后照样重传；重传包到达时接收方已收过 → 丢弃 + 回 ACK。**数据发了两遍，一遍是废的。** 所以 **RTO 估太短 = 无谓重传** —— 这正是自适应 RTO 与 Karn 算法要解决的问题。

**408 边界**：`snd_una` / `snd_nxt` 这类内核变量名不考，考的是「序号 = 字节编号」「确认号 = 期望的下一个字节」「累积确认」的语义；这里只为把「TCP 只维护一本字节账」这个直觉钉死。



## 传输层 · 块2 补讲：TCP 报文段的大小可控吗 —— MSS / MTU / 分片

**结论**：**不能「指定发多大」，但有一条硬上限可以协商。** TCP 面向字节流（块2 已证），段的大小由 TCP 自己切，应用无法指定「这次发 N 字节」。应用能动的只有上限和「催/憋」两个方向。

### 一、一个以太网帧的字节预算

| 帧头 | IP 首部 | TCP 首部 | **数据（= MSS）** | FCS |
|---|---|---|---|---|
| 14 B | 20 B | 20 B | **≤ 1460 B** | 4 B |

- **MTU = 1500 B**：链路层能承载的最大**载荷**（= IP 数据报最大长度），覆盖「IP 首部 + TCP 首部 + 数据」
- **MSS = MTU − 20(IP) − 20(TCP) = 1460 B**：**只对应数据部分**。报文段总长 = TCP 首部 + 数据 ≤ 1500
- 帧总长 1514 B（不含 8 B 前导码）

**MSS 的三条性质**（第 2 条最容易记反）：

1. **它是「上限」，不是「实际大小」。** 实际一段发多少，由**窗口 + 待发字节数 + 是否启用 Nagle** 决定，可能远小于 MSS（1 字节都可能）。
2. **MSS 选项通告的语义是「我最多能收多大」，不是「我要发多大」。** 只出现在 **SYN 段**（选项字段，Kind=2 / Len=4）。A 在 SYN 里告诉 B「我最多能收 MSS_A」→ 约束的是 **B 发过来的段**。每个方向各管各的，方向对称时等价于「取双方较小值」。
3. **握手时定，之后固定。** 未收到 MSS 选项时默认 **MSS = 536**（IPv4 最小重装缓冲 576 − 20 − 20）——**这个 536 必背**。

### 二、为什么必须要有 MSS，而不是「能装多少装多少」

因为 **IP 分片会放大丢失代价**：一个被切成 k 片的数据报，只要**丢 1 片**，整份数据报就作废——**重传时 k 片都要重发**。丢包率 p 下，分片把整包送达率从 (1−p) 压成 (1−p)^k。

所以 TCP 的策略是**从源头避免分片**：

- 用 **MSS 把段限制在 MTU 之内**（主动不给 IP 层分片的机会）
- **PMTUD（路径 MTU 发现）**：动态探出整条路径的最小 MTU，而不是只看第一跳

一句话：**MSS 存在的意义不是省首部开销（那是 Nagle 的活），而是避免分片带来的重传放大。**

### 三、谁能控制什么

| 控制点 | 谁在管 | 怎么动 | 典型值 |
|---|---|---|---|
| **单段数据上限 MSS** | 双方协商（SYN 选项） | 受本地 MTU 与对端通告值双重约束 | 以太网 1460；未协商默认 536 |
| **实际每段多大** | **TCP 栈自己** | 窗口 + 待发量 + Nagle | 0 ~ MSS |
| **MTU** | 链路层 | 硬件 / 配置（改不了电路径） | 以太网 1500 |
| 超 MTU 时 | IP 层 | IPv4 分片；DF 置位或 IPv6 则丢弃 + ICMP 报错（靠 PMTUD 收敛） | —— |

**应用程序真能动到的旋钮只有三个**（工程，408 不考）：

- `TCP_NODELAY`：禁用 Nagle → 小包立即发（降延迟，代价是小包变多）
- `TCP_CORK` / `MSG_MORE`：强制攒包，湊够 MSS 再发（降首部开销，代价是增延迟）
- `TCP_MAXSEG`：调 MSS，**基本只能调小**——上限还压着对端通告值和本地 MTU，调大无效

### 四、段大小 vs 窗口：两个独立旋钮

- **段大小（MSS）** 决定「一段能装多少字节」
- **窗口（cwnd / rwnd）** 决定「同时在途能有多少字节」

吞吐 ≈ **min(cwnd, rwnd) / RTT**，**与 MSS 无关**（只要窗口够大、段够满）。所以「MSS 小 = 慢」不成立——MSS 小只提高首部开销占比，不限制吞吐上限。

### 五、408 边界与术语校准

- **考**：MSS 的含义 / 位置（选项字段，仅 SYN）/ 与 MTU 的换算（`MSS = MTU − 40`）/ 默认 536；MTU 与 IP 分片（**片偏移单位 8 字节**、MF/DF 标志、**重组只在目的主机做**）
- **不考**：TCP_NODELAY / TCP_CORK / TCP_MAXSEG / TSO / GRO / PMTUD / IPv6 最小 MTU 1280
- **术语**：TCP 的 PDU 叫**报文段（segment）**，不是「包」；链路层是**帧**，网络层才是**分组 / 包**


## [CS144④] TCP 的三套序号与 ARQ（check2 + check3）

> 2026-10-20 · 来源：CS144 项目重修（learning-408/cs144 实读 + check0–7 官方 PDF）

- **我的理解**（Evidence-based）：TCP 在这套实现里被拆成两个方向的对偶件 —— check2 负责收端（乱序段 → 有序流），check3 负责发端（流 → 段 + 重传）。两端之间只靠 `TCPSenderMessage` / `TCPReceiverMessage` 两个纯数据结构接缝，所以能各自单测。

- **三套编号必须分清**（check2 PDF 的 'cat' 例子，★分清就是得分点）：seqno（32 位、从随机 ISN 起、**含** SYN/FIN、会回绕）/ absolute seqno（64 位、从 0 起、含 SYN/FIN）/ stream index（64 位、从 0 起、**不含** SYN/FIN）。换算：`stream_index = absolute_seqno - 1`；32↔64 的跨越靠 `Wrap32::wrap / unwrap(zero_point, checkpoint)`，checkpoint 取 first unassembled。

- **receiver 侧的两个信号**（`cs144` 骨架 `util/tcp_receiver_message.hh`）：`ackno`（optional，收到 SYN 之前为空）+ `window_size`（上限 65535 = UINT16_MAX）+ `RST`；window_size 的含义是「从 ackno 起我还愿意收多少个序号」。

- **sender 侧的计时六条**（check3 PDF §2.1）：禁止碰真实时钟（只能用 tick）；发出任何占序号空间的段时若 timer 未开则开；全部被确认则停表；到期重传**最早**那一个；若窗口非零则连续重传计数 +1 且 **RTO 翻倍**；收到新 ACK → RTO 复位回初值、有 outstanding 就重启 timer、连续重传计数清零。

- **考试侧重点**：★必考区。序号针对字节编号、SYN/FIN 各消耗一个序号、确认号的含义、ARQ 与超时重传、RTO 指数退避、**零窗口死锁与零窗口探测**。

- **易错点**：① `unwrap` 里拿有符号的 `abs()` 比距离（应该用无符号 mod 2^32 的距离比较）；② 第二个 SYN 到了就把 ISN 重设（不该）；③ 忘记把 window_size 截断到 65535；④ 零窗口时以为要把「假窗口 1」存进成员（只在 `push()` 内部假装一次，用来「戳」出一个新 ACK）；⑤ 把三段合并成一个大段重传（官方不允许）；⑥ 混淆 —— lab 里没有 cwnd，那里的窗口就是纯 rwnd。

- **边界**：check3 PDF 写 `MAX_PAYLOAD_SIZE` 是 1452，而实际骨架 `util/tcp_config.hh` 里是 1000（`TIMEOUT_DFLT = 1000`、`MAX_RETX_ATTEMPTS = 8`、`CAPACITY = 64000`）。**以自己 merge 到的骨架为准**；这两个数字不同不影响 408 答题。

- **复习计划**：R1 @ 2026-10-21 · R2 @ 2026-10-23 · R3 @ 2026-10-27


## [CS144⑤] check4：把你的 TCP 推到真实互联网上

> 2026-10-20 · 来源：CS144 项目重修（learning-408/cs144 实读 + check0–7 官方 PDF）

- **我的理解**（Evidence-based）：check4 几乎不写新代码，它是**验收**而不是开发 —— 回答的是「能不能通」「不通时错在哪」。

- **项目映射**（出处：`cs144/apps/tcp_ipv4.cc`、`cs144/util/tcp_over_ip.cc`、`cs144/src/tcp_minnow_socket.cc`）：调用链 `TunFD('tun144') → TCPOverIPv4OverTunFdAdapter → LossyFdAdapter → TCPMinnowSocket → bidirectional_stream_copy`（最后一层与 `tcp_native` 完全一致）。重点读 `wrap_tcp_in_ip()`：设 src/dst IP → 借 `ip_dgram.header.pseudo_checksum()` 算 TCP 校验和 → 序列化填进 payload → 再 `ip_dgram.header.compute_checksum()`。顺序不能反。

- **关键路径**：`./scripts/tun.sh start 144`（**每次重启 VM 都要重跑**）→ 把 `webget.cc` 里的 `TCPSocket` 换成 `CS144TCPSocket` → `cmake --build build --target check_webget`。

- **调试的黄金三角**：`tcp_ipv4 169.254.144.1 9090`（你的 TCP 做客户端）+ `tcp_native -l 0 9090`（内核 TCP 做服务端，参照组）+ `tshark -i tun144 -vvv`（站在网线上看包）。上层代码是同一套 → **跑不通，错误百分之百在你这边**。

- **考试侧重点**：连接管理（握手 / 挥手 / TIME_WAIT）的实战锚点。另外 check4 还要求一份 2 小时 ping 的统计报告 —— 那正好是课本里「丢包到底是 Bernoulli 还是突发」的一次实证。

- **易错点**：① 忘了跑 `tun.sh`（TUN 设备不存在）；② 调试信息打到 `cout`（污染 HTTP body，必须 `cerr`）；③ 算 TCP 校验和时漏掉伪首部 `pseudo_checksum()` → 对端**静默丢包**；④ ISN 没换成随机（`TCPConfig` 默认 `Wrap32{137}` 是给单测用的确定性值）；⑤ 忽视最后的 lingering（`TCPPeer::active()`：`10 * rt_timeout`，默认 10 s）→ 连接提前销毁、对端收到 RST。

- **待解决问题**：2 小时 ping 报告属复习以外的任务，优先级最低；真要做时按官方 10 问清单跑（丢包率 / 最长连续成功 / 最长突发丢失 / 条件概率 / RTT 时序图 / RTT 分布 / 相邻 RTT 相关性）。

- **复习计划**：R1 @ 2026-10-21 · R2 @ 2026-10-23 · R3 @ 2026-10-27


## [CS144·验收] 真题对位变式：连接管理与滑动窗口（§8 题型 1–4）

> 2026-10-22 · 来源：CS144 真题对位变式（自编，题目与解答真源 = `modules/408/CS144.md` §8.1；全部答案先手算复核再落盘）

- **用法（对齐 LEARNING.md §六）**：限时作答——选择 2 分钟、大题 15–20 分钟；先遮住【解】自己算，再对答案。卡住的题回 `modules/408/CS144.md` §3.2 / §3.3 / §3.4 回补，并按归因维度记账：概念不清 / 计算失误 / 没记住 / 审题错 / 项目误导 / 存疑待核。

**题型 1：握手与挥手的序号推算**（★核心 · 对位 check2「三套编号」+ check4 连接管理）

【题】主机 A 与 B 建立 TCP 连接，A 的 ISN = 1000，B 的 ISN = 3000。握手完成后 A 立即发送一段 100 字节的数据。求：
(1) B 的 SYN+ACK 段的 seq 与 ack；
(2) A 的第三次握手的 ACK 段的 seq 与 ack；
(3) A 首个数据段的 seq；B 收到该段后回的 ACK 号。

【模板】SYN 占 1 个号、FIN 占 1 个号、每个数据字节占 1 个号；**ack = 期望收到的下一个序号**；握手后数据起始 seq = 自己的 ISN + 1。

【解】B→A SYN+ACK：seq=**3000**，ack=**1001**｜A 第三条 ACK：seq=**1001**，ack=**3001**｜A 首个数据段：seq=**1001**（100 字节占 1001–1100）｜B 回 ACK：ack=**1101**。

【变式】A 再发 200 字节后关闭（发 FIN），B 不发数据直接关闭。求 A 的 FIN 的 seq、B 对该 FIN 的 ack、B 的 FIN 的 seq、A 对 B 的 FIN 的 ack。
【解】200 字节占 1101–1300 → A 的 FIN seq=**1301**，B 回 ack=**1302**；B 的 SYN 已占 3000 且无数据 → B 的 FIN seq=**3001**，A 回 ack=**3002**。

【易错】B 的 FIN 是 ISN+2 而不是 ISN+1——**SYN 占号**在握手方向上最容易被忽略；纯 ACK（第三次握手）不消耗序号，所以 A 的数据段 seq 与第三条 ACK 的 seq 相同（都是 1001）。

**题型 2：发送窗口 = min(rwnd, cwnd)**（★核心 · lab 与 408 的最大差异点）

【题】A 向 B 发送数据，B 通告 rwnd = 2000 字节。A 当前 snd_una = 1000，snd_nxt = 2500。
(1) 不考虑拥塞控制，A 还能发送多少字节？
(2) 若 B 的 ACK 把 snd_una 推进到 1500（rwnd 仍 2000），A 还能发多少？
(3) 若同时 cwnd = 1500（408 视角），发送窗口与还能发的字节数？

【模板】发送窗口 $= \min(\text{rwnd}, \text{cwnd})$；还能发 = 发送窗口 − 已发未确认 $= \min(\text{rwnd}, \text{cwnd}) - (\text{snd\_nxt} - \text{snd\_una})$。

【解】(1) 2000 − (2500−1000) = **500** 字节。(2) 2000 − (2500−1500) = **1000** 字节。(3) min(2000, 1500) = 1500；1500 − 1500 = **0** 字节。

【CS144 对位】check3 的 TCPSender **没有 cwnd**，lab 里只有 (1)(2) 的算法；408 真题必须先取 min。这是 `CS144.md` §2.8「没有拥塞控制」在考试侧的直接后果——**别把 lab 习惯带进考场**。

**题型 3：RTO 两种考法**（☆常规 · 课本加权平均 vs lab 指数退避）

【题】(1) 课本法：RTT 估计值 RTTs = 100 ms，新样本 SampleRTT = 200 ms，α = 1/8，求新 RTTs。
(2) lab 法：CS144 初始 RTO = 1000 ms，连续两次超时重传（窗口非零），RTO 依次变成多少？之后收到确认新数据的 ACK，RTO 变多少？
(3) 连续重传达到 MAX_RETX_ATTEMPTS = 8 时，上层的 TCPConnection 做什么？

【解】(1) RTTs_new = (1 − 1/8)×100 + (1/8)×200 = 87.5 + 25 = **112.5 ms**。
(2) 1000 → **2000** → **4000**（每次超时翻倍）；新 ACK 后**复位回初始值 1000**（不是减半）。
(3) 判定连接不可救，**放弃连接并把流出错**（对外表现为 RST，对端 stream set_error）。

【易错】(2) 里「RTO 翻倍」仅在窗口非零时伴随「连续重传计数 +1」一起发生；复位目标是**初始值**。课本的 RTO = RTTs + 4×RTTd 公式 408 也要会，但 CS144 没实现它——两边都别混。

**题型 4：零窗口与坚持探测**（★核心 · check3 明文规定）

【题】B 通告 window_size = 0。
(1) 408 术语：A 应启动什么、发什么？为什么不能干等？
(2) CS144 里 push() 怎么处理？这条规定的适用边界是什么？
(3) 接收窗口随后恢复为非零，连接为什么不会自动解锁？

【解】(1) 启动**坚持定时器**，周期性发送**零窗口探测报文**；(2) push() **假装窗口 = 1**，发一个可能被拒收的字节去「戳」出一个携带新窗口通告的 ACK——**假装只发生在本次 push 内部，不能把假窗口 1 存进成员**；(3) 接收方没有「窗口恢复」的主动通知，窗口通告只能**搭 ACK 的顺风车**——不探测就永远等不到新 ACK，死锁。

- **复习计划**：R1 @ 2026-10-23 · R2 @ 2026-10-25 · R3 @ 2026-10-29